# LangChain Model I/O: Prompt, Chat Model, Output Parser

Model I/O는 LangChain에서 다음 세 경계를 나눈다.

- `Prompt`: 데이터와 지시를 모델에 전달할 입력으로 만든다.
- `Chat Model`: 완성된 입력을 받아 모델을 호출한다.
- `Output Parser`: 모델 응답을 프로그램이 사용할 자료형으로 바꾼다.

세 단계를 분리하면 같은 모델을 사용해도 입력 형식, 대화 역할과 후속 데이터 처리 규칙을 각각 바꿀 수 있다.

Chat Model은 문자열 또는 message 목록을 받아 보통 `AIMessage`를 반환한다. 반환값은 필요한 정보의 범위에 따라 다르게 사용한다.

- `AIMessage.content`: 문자열 또는 text·reasoning·tool call 같은 content block 목록을 담는다.
- `AIMessage.text`: content에서 text block만 문자열로 꺼낸다.
- `AIMessage` 전체: reasoning, tool call, 사용량 또는 대화 이력을 다음 단계에 전달할 때 보존한다.

이번 노트북은 템플릿·예시·파서를 거쳐 `입력 딕셔너리 → Prompt → AIMessage → list/dict/Pydantic 객체` 흐름을 만든다.

<img src="https://d.pr/i/Wy5B5B+" width="1000" alt="LangChain 구성도에서 Model I/O 영역"/>

왼쪽 초록 영역은 Prompts, Language models, Output parsers가 연결되는 Model I/O를 가리킨다.

이번 단원은 이 영역만 다루며, 전체 그림에는 이전 세대 구성과 명칭이 섞일 수 있으므로 최신 API는 아래의 공식 Models, Prompt, structured output 문서와 코드로 확인한다.


## 환경 준비

- [LangChain Models](https://docs.langchain.com/oss/python/langchain/models)
- [Prompt templates](https://reference.langchain.com/python/langchain-core/prompts)
- [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)


## 1. Chat Model은 message를 받고 AIMessage를 돌려준다

Chat Model에는 입력 목적에 따라 다음 두 가지 형태를 전달한다.

- `문자열`: 독립적인 질문이나 지시처럼 대화 맥락이 필요하지 않을 때 사용한다.
- `message 목록`: 이전 대화나 역할별 지시를 함께 전달할 때 사용한다.

message 목록의 각 항목에는 텍스트의 역할이 지정된다.

- `system`: 모델이 따라야 할 행동 기준과 답변 방식을 정한다.
- `human` 또는 `user`: 사용자의 질문이나 요청을 나타낸다.
- `ai` 또는 `assistant`: 이전에 모델이 생성한 응답을 나타낸다.

역할은 단순한 라벨이 아니라 대화에서 각 텍스트를 해석하는 위치를 정한다.

`ChatOpenAI`는 `invoke()` 뒤 `AIMessage`를 반환한다. Responses API의 `content`에는 text·reasoning 같은 block이 함께 들어갈 수 있다.

- 답변 문자열만 표시할 때는 `.text`를 사용한다.
- 도구 호출, 사용량 또는 대화 이력을 이어 갈 때는 message 객체 전체를 보존한다.

자세한 구조는 [LangChain Messages](https://docs.langchain.com/oss/python/langchain/messages)에서 확인한다.


AIMessage
[{'id': 'rs_06d83914ee107d5d006a74269ca230819aa599f8d03a30429a', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqdCagOsmcyvJBTbLYjGPNlTU2abML---bsnr4NVj0popPQ7x5bIXG2lmHEOPjCDpn3VlByrZAkDrPBOzyHIEAHVQsNdVxBF3KlXdVtS3VsU0SCYnxYWtz_riZfBgLOPevUB60lfy0bw0JIuLXJuMBOhlhQuUA60av0EJrRUPbw2ZLyKnMbYuSM4Asck7kVeb3-s86HZcixom8Wi9lRpSkWGsJHzzf1bGSIyMBdsspXAycBFhdQT3eFBgXGaLHDWF3utO7RCcOwvghXd1JtvD5QDpRzVSifumoNcNCCLTpRimapFedUCKVuDuDE5XHyTyTiqI2FG9YB6vTjal9aca_0H3r7RcoIkqGlanqbQxblILA3qRLv1znzUFCkUBOSMqAl6sNY28e5C2AkHN_D4867IdQoUPxY0PKcoT9CMuzn8-sfupZRk4zwsiV4E7a5BjTknILYvl2UE6ZH1-Pk3KSr4vE4GNA-7LSpO3uuBVBuS5ct5Lk9R29iC0DqilX7zI4JlLg_7WuiVYwFtB0FPNhFyyd35j7AbvAds188m0AEdGpbZYDHfIewjY9syfe54YSzfiCk6dVbP6Scifhk0pDBdEazsX3bfG5zYf-zRhjo7k2WB8w2C705Zcw6tun8B1ljm84gf9MND-SuRJ7TpTk6RwVX1dn_GM_JATKtq5pL0Z3GIQ-m_xTPW3h5ul2vBYyt8sn4KTcGHKzJtlAVnzwKerNwOnVpI5QrvZIg1p1q4VjGWlWSjC6PyNRAatuEJMd36tDdo0DU1AxLT3cMFJp6B0DXh_yvJmtxMRnvesnA-q0v_KZaFJ1Wxre6KispvAIeWDecputhnOk

### 역할이 있는 message 목록 만들기

아래 셀은 API를 호출하지 않고 `SystemMessage`, `HumanMessage`, `AIMessage`가 각각 어떤 대화 위치를 차지하는지 확인한다. 실제 호출에서는 이 목록 전체를 `model.invoke(messages)`에 전달해 이전 응답까지 포함한 대화를 이어 갈 수 있다.


message의 타입을 확인합니다.
**************************************************
system: 모든 답변은 한 문장으로 작성하고 조선시대, 고려시대에서 쓸법한 문체로 답변한다
human: LangChain은 무엇인가요?
ai: LLM 애플리케이션의 구선 요소를 연결하는 프레임워크 입니다.
human: 그중 Model I/O의 역할은 무엇인가요?


### invoke, batch, stream은 언제 다른가

세 호출 방식은 입력 개수와 결과를 받는 시점이 다르다.

- `invoke()`: 입력 하나의 생성이 끝날 때까지 기다린 뒤 최종 `AIMessage` 하나를 받는다.
- `batch()`: 독립적인 여러 입력을 병렬로 처리하고 입력 순서에 맞는 `AIMessage` 목록을 받는다. LangChain의 클라이언트 측 병렬화이므로 공급자의 별도 Batch API와 다르다.
- `stream()`: 생성 중인 `AIMessageChunk`를 순서대로 받아 긴 답변의 첫 부분부터 화면에 보여 준다.
    - 청크 답변의 일부분을 나타냄 => 내용을 부분부분 나누는 것

공식 Models 문서의 호출 방식과 반환 차이를 기준으로, 요청끼리 문맥을 공유하면 `batch()`가 아니라 각 대화 이력을 따로 구성한다. parser가 완성된 JSON을 요구할 때는 chunk를 모두 합친 다음 파싱해야 한다.


invoke :  LangChain의 `batch()`는 하나의 체인이나 모델에 여러 입력을 한 번에 전달해 결과를 묶어서 반환하는 메서드입니다. 보통 독립적인 작업을 병렬 처리하므로 여러 요청을 개별적으로 호출하는 것보다 효율적입니다.

```python
results = chain.batch(["안녕", "날씨가 어때?", "LangChain이 뭐야?"])
print(results)
```


batch :  AIMessage LangChain의 Prompt는 LLM에 전달할 질문과 지시사항을 일정한 형식으로 구성하고 동적으로 관리하는 역할을 합니다.
batch :  AIMessage LangChain의 ChatModel은 대화형 언어 모델과 애플리케이션 간의 상호작용을 표준화해 메시지를 입력받고 응답을 생성하는 역할을 합니다.
batch :  AIMessage LangChain Output Parser는 LLM의 출력을 원하는 형식으로 변환하고 검증하는 역할을 합니다.


`LangChain ChatModel.stream()`은 모델 응답을 한 번에 기다리지 않고 **생성되는 토큰을 즉시 조금씩 전달**합니다.

장점:
- 첫 응답이 빨라 사용자 체감 대기 시간 감소
- 실시간 타이핑 UI 구현 가능
- 긴 응답도 점진적으로 표시해 사용자 경험 향상
- 생성 중 중단·처리 같은 제어가 쉬움AIMessageChunk


### Hugging Face와 ModelLaboratory의 위치

LangChain은 여러 모델 공급자를 같은 Runnable 흐름으로 연결할 수 있다. 공급자를 바꿔도 Prompt·Model·Parser의 경계는 그대로이다.

- `HuggingFacePipeline.from_model_id()`: 로컬 모델 다운로드와 실행 하드웨어가 필요하다.
- `HuggingFaceEndpoint`: 지원 모델, 인증 토큰, 이용 요금을 확인해야 한다.
- 최신 공급자 통합: [LangChain integrations](https://docs.langchain.com/oss/python/integrations/chat)에서 확인한다.

Model I/O 첫 학습에서는 무거운 모델을 실행하지 않고 공통 인터페이스와 선택 기준에 집중한다.

`ModelLaboratory`는 과거 여러 LLM의 응답을 비교할 때 사용하던 utility이다. 현재는 각 provider integration을 같은 입력으로 직접 호출해 비교한다. 이 방식은 결과·비용·지연 시간을 어떤 기준으로 비교하는지 코드에 명확히 남길 수 있다.


## 2. Prompt는 데이터에서 모델 입력을 만든다

두 Prompt template는 같은 변수 딕셔너리를 서로 다른 모델 입력으로 바꾼다.

- `PromptTemplate`: 변수 딕셔너리를 문자열 하나로 바꾼다. 한 문장 생성처럼 역할 구분이 필요 없는 입력에 적합하다.
- `ChatPromptTemplate`: 변수 딕셔너리를 역할이 있는 message 목록으로 바꾼다. system 지시, 사용자 질문과 이전 대화를 구분할 때 사용한다.

모델을 호출하기 전에 template의 `invoke()` 결과를 출력하면 변수 이름, 역할, 문장 순서를 무료로 검토할 수 있다. 이 확인은 실제 모델 호출 전에 입력 오류를 찾는 가장 짧은 방법이다.


### PromptTemplate의 변수 교체

`product` 값만 바꾸어도 동일한 문장 틀을 재사용할 수 있다. 출력은 `StringPromptValue`이며, 여기의 `.text`가 뒤에서 Chat Model에 들어갈 실제 문자열이다.


camera_prompt :  Compact Camera를 소개하는 광고 문구를 세 줄로 작성한다
fridge_prompt :  삼성 비스포크를 소개하는 광고 문구를 세 줄로 작성한다


### ChatPromptTemplate는 역할과 변수를 함께 관리한다

ChatPromptTemplate는 `domain`, `question` 같은 입력을 받아 `SystemMessage`, `HumanMessage` 순서의 `ChatPromptValue`를 만든다. 문자열 template에 system 지시를 섞어 쓰는 것보다, 어떤 문장이 행동 기준이고 어떤 문장이 사용자 질문인지 명확하다.


system : 당신은 AI Engineer 분야의 전문가 입니다
human : AI 엔지니어가 하는 일이 무엇인가?


### Few-shot은 답의 내용보다 답의 형식을 보여 준다

* few-shot : 질문과 답변을 함께 재공하여 프롬프트 포맷을 작성

Few-shot prompt는 모델에게 몇 개의 입력·출력 예시를 먼저 보여 준 뒤 새 입력을 준다.

목적은 모델의 일반 지식을 저장하는 것이 아니라, 이번 요청에서 따를 분류 라벨·답변 형식·추론 패턴을 구체적으로 보여 주는 데 있다.

좋은 예시는 다음 조건을 만족한다.

- 실제 입력과 같은 형식을 사용한다.
- 서로 다른 상황을 대표한다.
- 새 질문의 정답을 그대로 누설하지 않는다.

예시가 많아질수록 토큰 비용과 서로 충돌할 위험도 커지므로 최소 대표 예시부터 시작한다.


다음 수학 문제는 정답만 출력한다.

Q: 2 + 2 = ?
A: 4

Q: 3 + 5 = ?
A: 8

Q: 
    함수 f(x) = ln(1 + x)에 대해 다음을 순서대로 구해줘.

  1. f'(x)
  2. F(x) = ∫₀ˣ t·f'(t) dt
  3. lim(x → 0) [F(x) / x²]
  4. 수열 aₙ = n²[1/n - ln(1 + 1/n)]에 대해 lim(n → ∞) aₙ
   = ?
A:
1. \(\frac{1}{1+x}\)  
2. \(x-\ln(1+x)\)  
3. \(\frac{1}{2}\)  
4. \(\frac{1}{2}\)


## 3. Output Parser는 생성 텍스트를 후속 코드의 자료형으로 바꾼다

모델이 만든 텍스트는 사람이 읽기에는 충분해도 반복문, JSON 저장, API 응답에는 불편할 수 있다. Output Parser는 문자열 또는 `AIMessage`의 text 출력을 `list`, `dict` 같은 Python 객체로 변환해 다음 코드의 입력 계약을 만든다. parser는 형식을 검증할 뿐, 모델이 말한 사실이 참인지 검증하지는 않는다.

Parser는 필요한 Python 자료형에 따라 선택한다.

- `CommaSeparatedListOutputParser`: 쉼표로 구분된 텍스트를 Python 목록으로 바꾼다.
- `JsonOutputParser`: JSON 객체나 배열을 Python dict/list로 바꾼다.

두 parser 모두 모델에 출력 형식을 지시하는 단계와 반환값을 파싱하는 단계를 함께 설계해야 한다.


### parser에 문자열을 직접 넣기

먼저 API 호출 없이 parser 자체를 검증한다. 이 순서라면 오류가 생겼을 때 모델 생성 문제인지, 형식 지시 문제인지, parser 규칙 문제인지 분리할 수 있다.


<class 'list'> name :  list ['사과', '오랜지', '키위', '바나나', '두리안']


### JsonOutputParser로 JSON 계약을 확인하기

JSON 문자열은 사람이 읽기에는 명확해 보여도 Python에서는 아직 문자열이다. `JsonOutputParser`는 JSON 문법을 읽어 dict 또는 list로 바꾸며, 다음 코드가 `book["tags"]`처럼 key에 접근할 수 있게 한다.


{'title': 'LLM 입문', 'pages': 250, 'tags': ['AI', 'LangChain']}
dict list


### Prompt → Model → Parser를 LCEL로 연결하기

LCEL(LangChain Expression Language)의 `|` 연산자는 앞 단계의 출력을 다음 단계의 입력으로 연결한다. 아래 chain의 흐름은 `{"subject": ...} → StringPromptValue → AIMessage → list[str]`이다.

- `단계별 실행`: Prompt, Model과 Parser를 따로 호출하므로 오류가 발생한 위치를 확인하기 쉽다.
- `LCEL chain`: 같은 처리 흐름을 하나의 Runnable로 묶어 반복 실행하거나 다른 chain과 연결하기 쉽다.


['KIA 타이거즈: 한국시리즈 우승 11회로 전통과 성과가 뛰어남', '삼성 라이온즈: 한국시리즈 우승 8회로 강한 역사와 팬층을 보유함', '두산 베어스: 꾸준한 포스트시즌 진출과 안정적인 선수 육성으로 유명함', 'LG 트윈스: 서울 연고의 대형 인기 구단이며 최근 우승으로 경쟁력을 입증함', 'SSG 랜더스: 인천 연고의 강팀으로 창단 이후 우승을 경험하고 신흥 명문으로 자리 잡음']


## 4. Parser 방식과 provider-native structured output의 선택 경계

출력 구조가 단순한지, 필드 계약이 필요한지에 따라 방법을 선택한다.

- 문자열 parser: 쉼표 목록이나 이미 생성된 JSON 텍스트를 Python 자료형으로 바꿀 때 적합하다.
- provider-native structured output: 필드 이름과 타입이 반드시 맞아야 할 때 우선한다.

공식 LangChain 문서는 공급자가 schema를 직접 강제할 수 있을 때 더 신뢰할 수 있는 방법이라고 설명한다.

provider-native structured output에도 한계가 있다. 모델·provider의 지원 여부를 확인하고 schema를 올바르게 설계해야 한다. 형식은 보장해도 내용의 사실성까지 보장하지는 않는다.

provider-native schema를 지원하지 않으면 해당 chat-model 통합의 `function_calling` 등 structured-output 방식을 확인한다. 아래 코드는 `Prompt → Chat Model → Pydantic schema`를 결합하며, 실행 시 유료 API를 호출한다.


### Prompt → Model → structured output 연결하기

레시피 요청을 역할이 있는 prompt로 만들고, Pydantic `Recipe` schema를 `with_structured_output()`에 전달한다. model이 지원하는 provider-native structured output을 이용할 수 있으면 schema를 따르는 객체를 받고, 그렇지 않은 공급자는 그 통합의 structured-output 지원 방식을 확인해야 한다.


## 고객 문의 자동 분류·답변 초안 생성기

앞에서는 하나의 입력을 parser나 Pydantic 객체로 변환했다. 이제 고객 문의 여러 건으로 Model I/O의 전체 흐름을 확인한다.

- `Prompt`: 문의 내용과 분류 규칙을 모델 입력으로 만든다.
- `Chat Model과 `TicketAnalysis``: 문의를 처리하고 정해진 schema의 객체를 반환한다.
- `Python 후처리`: 객체를 DataFrame으로 바꾸고 기준값과 비교한다.

아직 Retrieval을 사용하지 않으므로 분류 규칙은 system message에 직접 제공한다. `batch()`는 여러 입력을 편리하게 처리하지만 공급자의 단일 Batch API가 아니라 독립적인 모델 요청을 병렬로 실행하므로 문의 건수만큼 API 사용량이 발생한다.


### 문의 데이터와 비교 기준 준비하기

각 문의에는 용도가 다른 두 종류의 값이 있다.

- `모델 입력`: `ticket_id`, `text`
- `결과 비교 기준`: `expected_category`, `expected_urgency`

기대값은 Prompt에 넣지 않고 모델 출력과 나중에 비교한다. 이 값은 분류 규칙이 적용되는지 확인하는 소규모 기준이며 일반화 성능을 추정하기 위한 홀드아웃 데이터와는 목적이 다르다.


### 반환 계약과 분류 기준 정의하기

`TicketAnalysis`는 모델이 반드시 채워야 할 필드와 허용 값을 정한다. 주요 필드의 역할은 다음과 같다.

- `category`, `urgency`: `Literal`로 허용된 문자열만 반환하게 제한한다.
- `summary`: 문의의 핵심을 한 문장으로 정리한다.
- `required_information`: 처리에 필요하지만 문의 원문에 없는 정보를 기록한다.
- `reply_draft`: 처리 완료를 단정하지 않는 고객용 답변 초안을 만든다.

schema는 필드의 존재와 자료형을 검증하지만 분류가 실제 기준과 맞는지는 보장하지 않는다. 따라서 다음 단계에서 기대값과 별도로 비교한다.


### 여러 문의를 batch로 처리하기

`batch()`를 실행할 때 확인할 값은 다음과 같다.

- `입력`: 모델에는 `ticket_id`와 `text`만 전달하고 기대값은 제외한다.
- `출력`: 입력 목록과 같은 순서의 `TicketAnalysis` 목록을 반환한다.
- ``max_concurrency=3``: 동시에 처리할 최대 요청 수이며 결과 개수나 API 호출 횟수를 줄이지 않는다.


### 구조화된 결과를 DataFrame으로 검증하기

Pydantic 객체를 `model_dump()`로 딕셔너리로 바꾸면 DataFrame 행으로 사용할 수 있다. category와 urgency의 기대값을 ticket_id로 연결한 뒤 일치 여부를 계산한다. schema 통과 여부와 분류 규칙의 일치 여부는 다른 문제이므로 둘을 함께 확인해야 한다. 여기서 계산하는 값은 5개 사례의 규칙 일치율이며 모델의 일반화 성능을 뜻하지 않는다.


## 회의록 업무 정리기

고객 문의 처리와 같은 Model I/O 구조를 회의록에 적용한다. 회의록 원문에서 요약, 결정 사항과 후속 업무를 추출하되 원문에 없는 담당자나 기한을 추측하지 않는다.

완성할 처리 흐름은 다음과 같다.

1. `MeetingAction`에 업무 내용, 담당자와 기한을 정의한다.
2. `MeetingAnalysis`에 meeting_id, 요약, 결정 사항, 후속 업무와 누락 정보를 정의한다.
3. 회의록만 근거로 사용하도록 `ChatPromptTemplate`을 작성한다.
4. `with_structured_output()`과 `batch()`로 두 회의록을 처리한다.
5. 중첩된 action item을 행 단위 DataFrame으로 평탄화한다.
6. `None`과 대표적인 누락 표현을 같은 값으로 정규화하고 추가 확인 대상으로 분리한다.


### 회의록 입력 준비하기

두 회의록은 누락 정보의 유무가 다르다.

- `첫 번째 회의록`: 모든 후속 업무에 담당자와 기한이 포함되어 있다.
- `두 번째 회의록`: 일부 업무의 담당자나 기한이 빠져 있다.

모델이 빠진 값을 지어내지 않고 `None`과 누락 정보로 표현하는지 확인한다.


### 회의록 업무 정리 코드 작성하기

위 처리 흐름을 하나의 코드셀로 완성한다. 고객 문의 예제의 `Prompt → Model → schema → DataFrame` 구조를 재사용하되, 중첩된 `action_items`를 업무별 행으로 펼치고 담당자나 기한이 없는 업무를 별도로 찾아야 한다.
